In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("IcebergDemo")
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.local.type", "hadoop")
    .config("spark.sql.catalog.local.warehouse", "/tmp/iceberg/warehouse")
    .getOrCreate()
)
print("Spark + Iceberg iniciado!")

26/05/05 21:02:14 WARN Utils: Your hostname, DESKTOP-9CLKLQP resolves to a loopback address: 127.0.1.1; using 172.19.75.57 instead (on interface eth0)
26/05/05 21:02:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/rafael/.cache/pypoetry/virtualenvs/spark-deltalake-iceberg-IhLNl8Fq-py3.11/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/rafael/.ivy2/cache
The jars for the packages stored in: /home/rafael/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c92f1cc7-96ad-4974-9062-86ff9a333df5;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 in central
downloading https://repo1.maven.org/maven2/org/apache/iceberg/iceberg-spark-runtime-3.5_2.12/1.5.0/iceberg-spark-runtime-3.5_2.12-1.5.0.jar ...
	[SUCCESSFUL ] org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0!iceberg-spark-runtime-3.5_2.12.jar (3114ms)
:: resolution report :: resolve 828ms :: artifacts dl 3117ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted

Spark + Iceberg iniciado!


In [2]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.ecommerce")

spark.sql("""
  CREATE TABLE IF NOT EXISTS local.ecommerce.clientes (
    id_cliente INT,
    nome STRING,
    email STRING,
    cidade STRING
  ) USING iceberg
""")

spark.sql("""
  CREATE TABLE IF NOT EXISTS local.ecommerce.pedidos (
    id_pedido INT,
    id_cliente INT,
    id_produto INT,
    quantidade INT,
    valor_total DOUBLE,
    status STRING,
    data_pedido DATE
  ) USING iceberg
""")

print("Tabelas Iceberg criadas!")

Tabelas Iceberg criadas!


In [7]:
from pyspark.sql import Row
from datetime import date

spark.sql("INSERT INTO local.ecommerce.clientes VALUES (1,'Ana Lima','ana@email.com','São Paulo')")
spark.sql("INSERT INTO local.ecommerce.clientes VALUES (2,'João Silva','joao@email.com','Curitiba')")

# Inserindo pedidos com data correta
from pyspark.sql.functions import to_date, lit
from pyspark.sql.types import *

pedidos_data = [
    (1, 1, 1, 1, 3500.00, 'pendente', date(2024, 1, 10)),
    (2, 2, 2, 2, 300.00,  'enviado',  date(2024, 1, 11)),
]

schema = StructType([
    StructField("id_pedido",   IntegerType(), False),
    StructField("id_cliente",  IntegerType(), False),
    StructField("id_produto",  IntegerType(), False),
    StructField("quantidade",  IntegerType(), False),
    StructField("valor_total", DoubleType(),  False),
    StructField("status",      StringType(),  False),
    StructField("data_pedido", DateType(),    False),
])

df_pedidos = spark.createDataFrame(pedidos_data, schema)
df_pedidos.writeTo("local.ecommerce.pedidos").append()

print("Dados inseridos!")
spark.sql("SELECT * FROM local.ecommerce.pedidos").show()

Dados inseridos!
+---------+----------+----------+----------+-----------+--------+-----------+
|id_pedido|id_cliente|id_produto|quantidade|valor_total|  status|data_pedido|
+---------+----------+----------+----------+-----------+--------+-----------+
|        1|         1|         1|         1|     3500.0|pendente| 2024-01-10|
|        2|         2|         2|         2|      300.0| enviado| 2024-01-11|
+---------+----------+----------+----------+-----------+--------+-----------+



In [8]:
spark.sql("UPDATE local.ecommerce.pedidos SET status = 'entregue' WHERE id_pedido = 1")
print("Após UPDATE:")
spark.sql("SELECT * FROM local.ecommerce.pedidos").show()

Após UPDATE:
+---------+----------+----------+----------+-----------+--------+-----------+
|id_pedido|id_cliente|id_produto|quantidade|valor_total|  status|data_pedido|
+---------+----------+----------+----------+-----------+--------+-----------+
|        2|         2|         2|         2|      300.0| enviado| 2024-01-11|
|        1|         1|         1|         1|     3500.0|entregue| 2024-01-10|
+---------+----------+----------+----------+-----------+--------+-----------+



In [9]:
spark.sql("DELETE FROM local.ecommerce.pedidos WHERE id_pedido = 2")
print("Após DELETE:")
spark.sql("SELECT * FROM local.ecommerce.pedidos").show()

Após DELETE:
+---------+----------+----------+----------+-----------+--------+-----------+
|id_pedido|id_cliente|id_produto|quantidade|valor_total|  status|data_pedido|
+---------+----------+----------+----------+-----------+--------+-----------+
|        1|         1|         1|         1|     3500.0|entregue| 2024-01-10|
+---------+----------+----------+----------+-----------+--------+-----------+



In [6]:
print("Snapshots disponíveis:")
spark.sql("SELECT * FROM local.ecommerce.pedidos.snapshots").show(truncate=False)

Snapshots disponíveis:
+-----------------------+-------------------+-------------------+---------+----------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                         |summary                                                                                                                                                                                                                 |
+-----------------------+-------------------+-------------------+---------+------------------------------------------------------------------------------